In [1]:
# Imports
import torch
import torch.nn as nn
import pandas as pd

In [3]:
sentiments = {
    "LABEL_0": "Bearish",
    "LABEL_1": "Bullish", 
    "LABEL_2": "Neutral"
}

In [4]:
# Load Train and Valid

train = pd.read_csv("sent_train.csv")
valid = pd.read_csv("sent_valid.csv")

In [5]:
# Check the are loaded correctly

display(train.head())
display(valid.head())

,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0


,text,label
0,$ALLY - Ally Financial pulls outlook https://t...,0
1,"$DELL $HPE - Dell, HPE targets trimmed on comp...",0
2,$PRTY - Moody's turns negative on Party City h...,0
3,$SAN: Deutsche Bank cuts to Hold,0
4,$SITC: Compass Point cuts to Sell,0


In [6]:
# X and y data separation

X_train = train["text"]
y_train = train["label"]

X_valid = valid["text"]
y_valid = valid["label"]

In [7]:
# LSTM Architecture
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        embedded = self.embedding(x)  
        output, (h_n, c_n) = self.lstm(embedded)
        
        pooled = output.mean(dim=1) # Mean pooling
        logits = self.fc(pooled)
        return logits

In [18]:
# Get every word
words = set()

# Get all types
for text in X_train:
    for word in text.lower().split():
        words.add(word)

# Map words to index
vocabulary = {w: idx for idx, w in enumerate(words)}

# Convert words to index
def words_to_idx(text):
    return [vocabulary.get(word, 0) for word in text.lower().split()]

In [19]:
# Get vocabulary size
vocab_size = len(vocabulary)

# Get number of classes
num_classes = y_train.nunique()

# Label to sentiment
def label_to_sentiment(label):
    if label == 0:
        return "Bearish"
    elif label == 1:
        return "Bullish"
    elif label == 2:
        return "Neutral"

# Embed dimensions    
embed_dim = 100

In [20]:
from torch.utils.data import Dataset, DataLoader

max_len = max(len(words_to_idx(text)) for text in X_train)

def words_to_tensor(text, max_len):
    idxs = words_to_idx(text)
    # pad with zeros if shorter than max_len
    if len(idxs) < max_len:
        idxs += [0] * (max_len - len(idxs))
    else:
        idxs = idxs[:max_len]  # truncate if too long
    return torch.tensor(idxs, dtype=torch.long)

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

X_train_tensors = [words_to_tensor(text, max_len) for text in X_train]
X_valid_tensors = [words_to_tensor(text, max_len) for text in X_valid]

train_dataset = TextDataset(X_train_tensors, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

valid_dataset = TextDataset(X_valid_tensors, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=True)

In [22]:
# Instance
model = LSTMClassifier(vocab_size=vocab_size, embed_dim=embed_dim, hidden_dim=64, num_classes=num_classes)

# Train our model
criterion = nn.CrossEntropyLoss() # Multiclass
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss+=loss.item()
    mean_loss = total_loss / len(train_loader)
    
    # Validation
    model.eval()  
    correct = 0
    total = 0

    with torch.no_grad():  # no gradient calculation during validation
        for X_batch, y_batch in valid_loader:
            logits = model(X_batch)                  # (batch_size, num_classes)
            predictions = torch.argmax(logits, dim=1)  # choose class with highest logit
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    val_acc = correct / total  # validation accuracy

    print(f"Epoch {epoch+1} | Train Loss: {mean_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1 | Train Loss: 0.8357 | Val Acc: 0.7039
Epoch 2 | Train Loss: 0.6510 | Val Acc: 0.7471
Epoch 3 | Train Loss: 0.4781 | Val Acc: 0.7776
Epoch 4 | Train Loss: 0.3205 | Val Acc: 0.7936
Epoch 5 | Train Loss: 0.2026 | Val Acc: 0.7952
Epoch 6 | Train Loss: 0.1237 | Val Acc: 0.7915
Epoch 7 | Train Loss: 0.0761 | Val Acc: 0.7940
Epoch 8 | Train Loss: 0.0564 | Val Acc: 0.7831
Epoch 9 | Train Loss: 0.0398 | Val Acc: 0.7831
Epoch 10 | Train Loss: 0.0289 | Val Acc: 0.7743


In [27]:
import random

# Pick a random index
idx = random.randint(0, len(X_valid_tensors) - 1)

sample_text_tensor = X_valid_tensors[idx].unsqueeze(0)  # add batch dim: (1, max_len)
true_label = y_valid[idx]

print("Text:", X_valid[idx])
print("True label:", true_label)

model.eval()  # set to evaluation mode
with torch.no_grad():
    logits = model(sample_text_tensor)               # (1, num_classes)
    predicted_class = torch.argmax(logits, dim=1).item()

print("Predicted label:", predicted_class)

print("Sentiment: ", label_to_sentiment(predicted_class))

Text: $TIF: Tiffany & Co confirms agreement to be acquired by LVMH (LVMUY) for $135/share in cash, or approximately $16.2… https://t.co/xhNzqZ8PVx
True label: 2
Predicted label: 2
Sentiment:  Neutral
